# TPU-Gen POC

User prompt -> prompt_formatter -> LLM -> `.vh` -> retrieval -> full `.v`
-> Yosys (inside ORFS) -> OpenROAD -> GDSII + PPA, optionally for several rounds.

The backend is OpenROAD only. If the ORFS image is missing the flow raises;
no PPA number in this notebook comes from anywhere else.
See `plan.md` for the design and `lib/` for the modules.


In [ ]:
import os
import sys
sys.path.insert(0, "lib")

import paths
from flow import TPUGenFlow, MaxIterations, NoErrors, PPAWithinTarget
from llm import get_backend
from openroad import DEFAULT_IMAGE, ORFSConfig, check_image
from tpugen_types import PPATarget

paths.check()
OPENROAD_IMAGE = DEFAULT_IMAGE
check_image(OPENROAD_IMAGE)
print("upstream RTL:", paths.RTL_DIR)

## 1. The request

In [ ]:
user_prompt = (
    "Generate a 4x4 systolic array TPU, INT8 datapath, "
    "using the DRUM_APTPU approximate multiplier and the APPROX5 "
    "approximate adder, with a low precision coefficient of 4."
)

# Area and power follow the paper's LeNet 4x4 INT8 row. WNS is the slack we
# want left over at the flow's clock period, not the paper's delay column.
target = PPATarget(area=32208, wns=0.5, total_power=0.036)

## 2. What prompt_formatter produces

The fine-tuned model was trained on Description/Metrics pairs, so the request
is rewritten into exactly that shape before it reaches the LLM.

In [ ]:
from prompt_formatter import format_prompt, parse_user_prompt

spec = parse_user_prompt(user_prompt)
print(spec)
print()
print(format_prompt(spec, target))

## 3. Pick the LLM backend

`get_backend` reads from the backend registry in `lib/llm/registry.py`. Two of
the entries are local coding-agent CLIs driven one-shot, so they need the CLI
signed in but no API key of our own:

| name | what runs |
| --- | --- |
| `claude` | `claude -p` (Claude Code CLI) |
| `codex` | `codex exec` (OpenAI Codex CLI) |

Set `AGENT` below to `"claude"` or `"codex"`. `"replay"` is the offline
baseline: a nearest neighbour lookup in the training set, no model involved.

In [ ]:
from llm import available_backends, describe_backends

AGENT = "codex"           # <- "claude" | "codex" | "replay" | "anthropic" | "openai"
MODEL = "gpt-5.6-luna" if AGENT == "codex" else None  # verified with this host's Codex CLI

for name, what in describe_backends().items():
    print(f"  {'*' if name == AGENT else ' '} {name:<10} {what}")

if AGENT not in available_backends():
    raise SystemExit(f"{AGENT!r} is not registered")

llm = get_backend(AGENT, **({"model": MODEL} if MODEL else {}))
print(f"\nusing: {llm.name}")

## 4. Run the flow

Every round is a full place and route driven by the backend picked above.


In [ ]:
orfs = ORFSConfig(
    image=OPENROAD_IMAGE,
    clock_period=5.0,        # ns
    core_utilization=30,
)

tpugen = TPUGenFlow(llm, orfs=orfs, retrieval_mode="rag")

# Each round is a full place and route. On a 2-core machine a 4x4 INT8 design
# takes tens of minutes per round, so keep the round count small.
result = tpugen.run(
    user_prompt,
    target,
    stop=PPAWithinTarget(target, tolerance=0.25),
    max_iterations=2,
)

print(result.table())
print("\nstopped:", result.stopped_because)
print("run dir:", result.run_dir)

## 5. The design that came out

In [ ]:
best = result.best
if best is None:
    raise SystemExit("no iteration reached GDSII; inspect runs/<id>/iter_*/orfs.log")

print("GDSII :", best.gds, f"({best.gds.stat().st_size/1e6:.1f} MB)")
print("area  :", f"{best.ppa.area:.1f} um2")
print("WNS   :", f"{best.ppa.wns:.3f} ns")
print("power :", f"{best.ppa.total_power:.3e} W")
print("source:", best.ppa.source)
print()
for k, v in best.ppa.raw.items():
    print(f"  {k:20} {v}")

## 6. What retrieval saved

The paper's claim is that RAG fetches only the modules a header actually
needs. `mode="full"` hands the whole library to OpenROAD instead, so the two
can be compared directly.

In [ ]:
import json

from retrieval import RTLLibrary, retrieve

library = RTLLibrary.load(paths.RTL_DIR)
vh = best.vh_path.read_text()

rag = retrieve(vh, library, paths.TOP_MODULE, mode="rag")
full = retrieve(vh, library, paths.TOP_MODULE, mode="full")

print(f"rag  : {len(rag.filelist):>3} files")
print(f"full : {len(full.filelist):>3} files")
print()
print("selected:", ", ".join(p.name for p in rag.filelist))

## 7. Iterating on PPA

`PPAWithinTarget` stops as soon as a round lands inside the tolerance;
otherwise the measured OpenROAD numbers and any tool errors go back into the
next prompt. Section 4 already ran that loop.

One caveat worth being explicit about: the `replay` backend is a nearest
neighbour lookup over the training set. The feedback does reach the prompt and
does change which entry is nearest, so later rounds return a different header
and different PPA -- but that is token overlap shifting, not the design being
steered. Only a real LLM backend can move the design toward the target.

In [ ]:
# The loop above already feeds the measured OpenROAD PPA of each round back
# into the next prompt. To chase a target harder, widen the budget:
#
# result = tpugen.run(
#     user_prompt,
#     target,
#     stop=PPAWithinTarget(target, tolerance=0.1),
#     max_iterations=5,
# )
#
# Every round is preserved under runs/<run-id>/iter_NN/, so the rounds can be
# compared after the fact:
for r in result.iterations:
    print(r.iteration, r.workdir.name, "ok" if r.ok else r.errors[:1])